In [1]:
import matplotlib.pyplot as plt
import numpy as np
#import geopandas as gpd
import pandas as pd
import os
import copy
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score
import dask.dataframe as dd
import lightgbm as lgb
from dask.diagnostics import ProgressBar
from pathlib import Path
import duckdb

## For select dates and large data

In [2]:
def make_predictions_dask(parquet_in, model_path, parquet_out, feature_cols):
    """
    Stage 1: Read Parquet, run LightGBM predictions, save with 'prediction' column.
    """
    Path(parquet_out).mkdir(parents=True, exist_ok=True)

    print("Loading model…")
    if model_path.endswith(".txt"):
        model = lgb.Booster(model_file=model_path)
    else:
        model = joblib.load(model_path)

    print("Reading data with Dask…")
    ddf = dd.read_parquet(parquet_in)
    ddf = ddf.rename(columns={'daily_NDVI': 'NDVI'})
    existing_features = [col for col in feature_cols if col in ddf.columns]
    ddf[existing_features] = ddf[existing_features].map_partitions(lambda df: df.round(2))

    print("Running predictions…")
    def predict_partition(part):
        part["prediction"] = model.predict(part[existing_features])
        return part

    ddf_pred = ddf.map_partitions(predict_partition)

    print(f"Saving predictions to {parquet_out} …")
    with ProgressBar():
        ddf_pred.to_parquet(
            parquet_out,
            write_index=False,
            compression="zstd",
            engine="pyarrow",
        )

    print("✅ Stage 1 complete — predictions saved.")

In [3]:
def scale_predictions_dask(parquet_with_preds, parquet_out_scaled):
    """
    Stage 2: Read Parquet with 'prediction' + 'NDVI',
    scale prediction to NDVI range, save new Parquet.
    """
    Path(parquet_out_scaled).mkdir(parents=True, exist_ok=True)

    print("Reading predicted Parquet with Dask…")
    ddf = dd.read_parquet(parquet_with_preds)

    # ---- Compute global NDVI/pred min–max (fast, one pass) ----
    print("Computing NDVI/pred range (DuckDB, fast)…")
    pq_path = str(Path(parquet_with_preds).absolute())
    try:
        query = f"""
        SELECT
            MIN(NDVI) AS ndvi_min,
            MAX(NDVI) AS ndvi_max,
            MIN(prediction) AS pred_min,
            MAX(prediction) AS pred_max
        FROM parquet_scan('{pq_path}/*.parquet')
        """
        ndvi_min, ndvi_max, pred_min, pred_max = duckdb.sql(query).fetchone()
    except Exception as e:
        print("DuckDB failed, falling back to Dask .agg() method:", e)
        return

    print(f"NDVI: [{ndvi_min:.4f}, {ndvi_max:.4f}] | Prediction: [{pred_min:.4f}, {pred_max:.4f}]")

    # ---- Scale predictions ----
    def scale_partition(df_part, pred_min, pred_max, ndvi_min, ndvi_max):
        df_part["prediction_scaled"] = np.nan
        valid_mask = (
            df_part["NDVI"].notnull() &
            (df_part["NDVI"] >= ndvi_min) &
            (df_part["NDVI"] <= ndvi_max)
        )
        df_part.loc[valid_mask, "prediction_scaled"] = (
            pred_min
            + ((df_part.loc[valid_mask, "NDVI"] - ndvi_min)
               / (ndvi_max - ndvi_min))
            * (pred_max - pred_min)
        )
        return df_part

    ddf_scaled = ddf.map_partitions(scale_partition, pred_min, pred_max, ndvi_min, ndvi_max)
    
    # ---- Save ----
    print(f"Saving scaled predictions to {parquet_out_scaled} …")
    with ProgressBar():
        ddf_scaled.to_parquet(
            parquet_out_scaled,
            write_index=False,
            compression="zstd",
            engine="pyarrow",
        )

    print("✅ Stage 2 complete — scaled predictions saved.")

In [4]:
def scale_predictions_dask_daily(parquet_with_preds, parquet_out_scaled):
    """
    Stage 2: Aggregate hourly predictions to daily values,
    then scale daily predictions linearly with NDVI
    (NDVI controls the relative position within the prediction range,
    but the overall prediction range is preserved).
    """
    Path(parquet_out_scaled).mkdir(parents=True, exist_ok=True)

    print("Reading predicted Parquet with Dask…")
    ddf = dd.read_parquet(parquet_with_preds)

    # ---- Ensure datetime format ----
    if "datetime" in ddf.columns:
        ddf["datetime"] = dd.to_datetime(ddf["datetime"])

    # ---- Filter valid hours (6 AM–6 PM) ----
    if "datetime" in ddf.columns:
        print("Filtering hours between 6 and 18 …")
        ddf = ddf[(ddf["datetime"].dt.hour >= 6) & (ddf["datetime"].dt.hour <= 18)]

    # ---- Clamp negatives ----
    if "prediction" in ddf.columns:
        print("Clamping negative predictions to 0 …")
        ddf["prediction"] = ddf["prediction"].clip(lower=0)

    # ---- Aggregate to daily sums ----
    print("Aggregating hourly predictions to daily values …")
    ddf_daily = (
    ddf.groupby(["latitude", "longitude"])
       .agg({"prediction": "sum", "NDVI": "mean"})
       .reset_index()
)

    with ProgressBar():
        df_daily = ddf_daily.compute()

    print(f"Aggregated {len(df_daily):,} daily pixel values.")

    # ---- Compute NDVI/pred min–max ----
    print("Computing NDVI/pred range (DuckDB, fast)…")
    pq_path = Path(parquet_with_preds).as_posix()
    try:
        query = f"""
        SELECT
            MIN(NDVI) AS ndvi_min,
            MAX(NDVI) AS ndvi_max,
            MIN(prediction) AS pred_min,
            MAX(prediction) AS pred_max
        FROM parquet_scan('{pq_path}/*.parquet')
        """
        ndvi_min, ndvi_max, pred_min, pred_max = duckdb.sql(query).fetchone()
    except Exception as e:
        print("DuckDB failed, falling back to Dask .agg() method:", e)
        ndvi_min, ndvi_max = df_daily["NDVI"].min(), df_daily["NDVI"].max()
        pred_min, pred_max = df_daily["prediction"].min(), df_daily["prediction"].max()

    print(f"NDVI: [{ndvi_min:.4f}, {ndvi_max:.4f}] | Daily Prediction: [{pred_min:.4f}, {pred_max:.4f}]")

    # ---- Scale predictions linearly based on NDVI ----
    print("Scaling daily predictions linearly with NDVI …")

    ndvi_norm = (df_daily["NDVI"] - ndvi_min) / (ndvi_max - ndvi_min)
    ndvi_norm = ndvi_norm.clip(0, 1)
    df_daily["prediction_scaled"] = pred_min + ndvi_norm * (pred_max - pred_min)

    print("Scaling complete. Range preserved.")

    # ---- Save ----
    print(f"Saving NDVI-scaled daily predictions to {parquet_out_scaled} …")
    df_daily.to_parquet(
        parquet_out_scaled,
        write_index=False,
        compression="zstd",
        engine="pyarrow",
    )

    print("✅ Stage 2 complete — NDVI-scaled daily predictions saved.")

## PA

In [5]:
lgm_model_path = os.path.join(os.path.dirname(os.getcwd()), '7_MLModel', 'ScenarioB_lgbm_final_model_001_PA.txt')
input_features_list = ['NDVI',
 'Air Temperature',
 'relative_humidity',
 'Downward Short-Wave Radiation Flux']

field_extent_parquet_dir = os.path.join(os.path.dirname(os.getcwd()), '8_Data_prepration_prediction_phase', 'For_select_dates', 'PA')

In [6]:
for file in os.listdir(field_extent_parquet_dir):
    if file.endswith('met'):
        fe_dir = os.path.join(field_extent_parquet_dir, file)
        make_predictions_dask(
            parquet_in= fe_dir,
            model_path=lgm_model_path,
            parquet_out=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            feature_cols=input_features_list
        )
        scale_predictions_dask(
            parquet_with_preds=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            parquet_out_scaled=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions_scaled'))
        )

Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\PA\GBF_20230518_ndvi_hourly_with_predictions …
[########################################] | 100% Completed | 17m 11s
✅ Stage 1 complete — predictions saved.
Reading predicted Parquet with Dask…
Computing NDVI/pred range (DuckDB, fast)…
NDVI: [0.2600, 0.9300] | Prediction: [0.0296, 0.6685]
Saving scaled predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\PA\GBF_20230518_ndvi_hourly_with_predictions_scaled …
[########################################] | 100% Completed | 8.98 ss
✅ Stage 2 complete — scaled predictions saved.
Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\C

## CA

In [7]:
lgm_model_path = os.path.join(os.path.dirname(os.getcwd()), '7_MLModel', 'ScenarioB_lgbm_final_model_001_CA.txt')
input_features_list = ['NDVI',
 'Air Temperature',
 'relative_humidity',
 'Downward Short-Wave Radiation Flux']

field_extent_parquet_dir = os.path.join(os.path.dirname(os.getcwd()), '8_Data_prepration_prediction_phase', 'For_select_dates', 'CA')

In [8]:
for file in os.listdir(field_extent_parquet_dir):
    if file.endswith('met'):
        fe_dir = os.path.join(field_extent_parquet_dir, file)
        make_predictions_dask(
            parquet_in= fe_dir,
            model_path=lgm_model_path,
            parquet_out=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            feature_cols=input_features_list
        )
        scale_predictions_dask(
            parquet_with_preds=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            parquet_out_scaled=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions_scaled'))
        )

Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\CA\Bi1_20240609_ndvi_hourly_with_predictions …
[########################################] | 100% Completed | 26m 49s
✅ Stage 1 complete — predictions saved.
Reading predicted Parquet with Dask…
Computing NDVI/pred range (DuckDB, fast)…
NDVI: [0.0800, 0.9600] | Prediction: [0.0163, 1.0185]
Saving scaled predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\CA\Bi1_20240609_ndvi_hourly_with_predictions_scaled …
[########################################] | 100% Completed | 8.54 ss
✅ Stage 2 complete — scaled predictions saved.
Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\C

## IL-INm

In [9]:
lgm_model_path = os.path.join(os.path.dirname(os.getcwd()), '7_MLModel', 'ScenarioB_lgbm_final_model_001_IL.txt')
input_features_list = ['NDVI',
 'Air Temperature',
 'relative_humidity',
 'Downward Short-Wave Radiation Flux']

field_extent_parquet_dir = os.path.join(os.path.dirname(os.getcwd()), '8_Data_prepration_prediction_phase', 'For_select_dates', 'IL_IN')

In [10]:
for file in os.listdir(field_extent_parquet_dir):
    if file.endswith('met'):
        fe_dir = os.path.join(field_extent_parquet_dir, file)
        make_predictions_dask(
            parquet_in= fe_dir,
            model_path=lgm_model_path,
            parquet_out=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            feature_cols=input_features_list
        )
        scale_predictions_dask(
            parquet_with_preds=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions')),
            parquet_out_scaled=os.path.join(field_extent_parquet_dir, file.replace('_met', '_predictions_scaled'))
        )

Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\IL_IN\UiABC_20240607_ndvi_hourly_with_predictions …
[########################################] | 100% Completed | 12m 50s
✅ Stage 1 complete — predictions saved.
Reading predicted Parquet with Dask…
Computing NDVI/pred range (DuckDB, fast)…
NDVI: [0.2200, 0.9300] | Prediction: [0.0353, 0.5785]
Saving scaled predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Projects\ET\CONUS_sites_version\8_Data_prepration_prediction_phase\For_select_dates\IL_IN\UiABC_20240607_ndvi_hourly_with_predictions_scaled …
[########################################] | 100% Completed | 6.30 ss
✅ Stage 2 complete — scaled predictions saved.
Loading model…
Reading data with Dask…
Running predictions…
Saving predictions to c:\Users\adadkhah\OneDrive - University of Vermont\UVM\Pro